In [ ]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数（XGBoost 机器学习因子示例）

    ⚠️ 赛制约定（重要）:
        评测时平台只会替换 datasources / start_date / end_date 三个入参,且 start_date~end_date
        指向的是【测试集区间】。因此训练区间必须在代码里写死（见下方 TRAIN_START/TRAIN_END）,
        本函数只用写死的训练区间拟合模型,再用平台传入的测试区间做【样本外预测】。
        切勿用传入的 start_date/end_date 训练模型——那等于在测试集上训练,既是数据泄漏, 也无法体现因子真实的样本外能力。
        后期会审查这类问题

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}，平台会在公榜/私榜自动切换。
                            可用逻辑名: "bar1m" -> 分钟 K 线表, "financial" -> 财务数据表
        start_date (str): 测试集开始时间（平台注入）
        end_date (str):   测试集结束时间（平台注入）

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb

    # ============================================================
    # 训练区间：写死，不随平台入参变化
    # ============================================================
    TRAIN_START = '2020-01-01 00:00:00'
    TRAIN_END = '2023-12-31 23:59:59'

    feature_cols = ['roe', 'reversal_1', 'momentum_5', 'vol_20', 'hl_range', 'log_amount']

    # ============================================================
    # 特征工程：给定任意区间，构造特征 + 标签（训练/预测共用同一套逻辑）
    # ============================================================
    def build_features(sd, ed):
        bar1m_table = datasources['bar1m']
        financial_table = datasources['financial']

        # ----- 财务因子 ROE（公告日 PIT，向前多取 1 年保证起点有值可填充）-----
        fin_start = pd.to_datetime(sd) - pd.Timedelta(days=365)
        fin_sql = f"""
        WITH ttm AS (
            SELECT date, instrument, net_profit_to_parent_shareholders AS np_ttm
            FROM {financial_table} WHERE category='ttm' AND shift=0
        ),
        lf AS (
            SELECT date, instrument, total_equity_to_parent_shareholders AS equity_lf
            FROM {financial_table} WHERE category='lf' AND shift=0
        )
        SELECT date, instrument, np_ttm / equity_lf AS roe
        FROM ttm PRUNE JOIN lf USING (date, instrument)
        """
        fin = dai.query(fin_sql, filters={'date': [fin_start, ed]}).df()

        # ----- 量价因子（SQL 内日内聚合，分钟 -> 日频；向前多取 40 天作时序缓冲）-----
        price_start = pd.to_datetime(sd) - pd.Timedelta(days=40)
        price_sql = f"""
        SELECT
            CAST(strftime(date, '%Y-%m-%d') AS DATETIME)      AS date,
            instrument,
            SUM(amount) / NULLIF(SUM(volume), 0)              AS vwap,         -- 成交均价
            log(SUM(amount) + 1)                              AS log_amount,   -- 流动性
            (MAX(high) - MIN(low))
                / NULLIF(SUM(amount) / NULLIF(SUM(volume), 0), 0) AS hl_range  -- 日内振幅
        FROM {bar1m_table}
        WHERE close > 0                                                        -- 剔除集合竞价/停牌
        GROUP BY date, instrument
        """
        price = dai.query(
            price_sql, filters={'date': [price_start, ed]}, compression=True
        ).df()
        price['instrument'] = price['instrument'].astype(str)
        price = price.sort_values(['instrument', 'date']).reset_index(drop=True)

        # 基于日频 VWAP 衍生时序因子
        g = price.groupby('instrument', group_keys=False)['vwap']
        price['reversal_1'] = -g.pct_change(1)              # 1 日反转
        price['momentum_5'] = g.pct_change(5)               # 5 日动量
        price['vol_20'] = g.pct_change(1).groupby(price['instrument']).transform(
            lambda s: s.rolling(20, min_periods=5).std()    # 20 日波动率
        )
        # 标签：未来 1 日 VWAP 收益
        price['label'] = g.transform(lambda s: s.shift(-1) / s - 1)

        # ----- 合并财务（按公告日前向填充到日频）-----
        df = pd.merge_asof(
            price.sort_values('date'),
            fin.sort_values('date'),
            on='date', by='instrument',
        )

        # ----- 截面预处理：去极值 + 标准化 -----
        def winsorize_zscore(s):
            med, mad = s.median(), (s - s.median()).abs().median()
            if mad:
                s = s.clip(med - 5 * 1.4826 * mad, med + 5 * 1.4826 * mad)
            std = s.std()
            return (s - s.mean()) / std if std else s - s.mean()

        for col in feature_cols:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df.groupby('date', group_keys=False)[col].transform(winsorize_zscore)
            df[col] = df[col].fillna(0.0)

        # 只返回落在目标区间 [sd, ed] 内的行（前面多取的缓冲期只用于算特征，不输出）
        df = df[(df['date'] >= pd.to_datetime(sd)) & (df['date'] <= pd.to_datetime(ed))]
        return df.reset_index(drop=True)

    # ============================================================
    # 第 1 步：用【写死的训练区间】拟合模型
    # ============================================================
    train_df = build_features(TRAIN_START, TRAIN_END)
    train_df['label'] = train_df['label'].replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=['label'])  # 末尾无未来收益的行不参与训练

    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        reg_lambda=1.0, n_jobs=-1, random_state=42,
    )
    model.fit(train_df[feature_cols], train_df['label'])

    # ============================================================
    # 第 2 步：用【平台传入的测试区间】做样本外预测，预测值即为因子
    # ============================================================
    test_df = build_features(start_date, end_date)
    test_df['factor'] = model.predict(test_df[feature_cols])

    # ============================================================
    # 第 3 步：对齐中证 1000 成分股并输出
    # ============================================================
    # 只保留当日属于成分股的标的。bigalpha_2026_instruments 无需替换。
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    result = pd.merge(test_df[['date', 'instrument', 'factor']], stk_pool,
                      how='inner', on=['date', 'instrument'])
    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    return result.dropna(subset=['factor']).reset_index(drop=True)[['date', 'instrument', 'factor']]


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    # 本地用这段区间模拟「平台注入的测试集区间」（训练区间已在 main 内写死）
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_factorminer._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
